In [2]:
# ==========================================
# SISTEM FUZZY LOGIC REKOMENDASI PRODUK
# ==========================================

import pandas as pd
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# 1. =======================
# Load dataset
# ==========================
data = pd.read_csv("C:/Users/acer/fuzzy_recommendation_dataset.csv")

# 2. =======================
# Definisikan variabel fuzzy
# ==========================
price = ctrl.Antecedent(np.arange(0, 2000000, 10000), 'price')
rating = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'rating')
popularity = ctrl.Antecedent(np.arange(0, 1001, 10), 'popularity')
discount = ctrl.Antecedent(np.arange(0, 101, 5), 'discount')

recommendation = ctrl.Consequent(np.arange(0, 11, 1), 'recommendation')

# 3. =======================
# Membership Functions
# ==========================
# Harga
price['low'] = fuzz.trimf(price.universe, [0, 0, 800000])
price['medium'] = fuzz.trimf(price.universe, [500000, 1000000, 1500000])
price['high'] = fuzz.trimf(price.universe, [1000000, 2000000, 2000000])

# Rating
rating['low'] = fuzz.trimf(rating.universe, [0, 0, 2.5])
rating['medium'] = fuzz.trimf(rating.universe, [2.0, 3.5, 4.0])
rating['high'] = fuzz.trimf(rating.universe, [3.5, 5.0, 5.0])

# Popularitas
popularity['low'] = fuzz.trimf(popularity.universe, [0, 0, 300])
popularity['medium'] = fuzz.trimf(popularity.universe, [200, 500, 700])
popularity['high'] = fuzz.trimf(popularity.universe, [600, 1000, 1000])

# Diskon
discount['low'] = fuzz.trimf(discount.universe, [0, 0, 20])
discount['medium'] = fuzz.trimf(discount.universe, [10, 30, 50])
discount['high'] = fuzz.trimf(discount.universe, [40, 100, 100])

# Output: Rekomendasi
recommendation['low'] = fuzz.trimf(recommendation.universe, [0, 0, 5])
recommendation['medium'] = fuzz.trimf(recommendation.universe, [3, 5, 7])
recommendation['high'] = fuzz.trimf(recommendation.universe, [6, 10, 10])

# 4. =======================
# Fuzzy Rules
# ==========================
rule1 = ctrl.Rule(rating['high'] & popularity['high'], recommendation['high'])
rule2 = ctrl.Rule(rating['medium'] & popularity['medium'], recommendation['medium'])
rule3 = ctrl.Rule(price['high'] & discount['low'], recommendation['low'])
rule4 = ctrl.Rule(price['low'] & rating['high'], recommendation['high'])
rule5 = ctrl.Rule(discount['high'] & popularity['medium'], recommendation['high'])
rule6 = ctrl.Rule(price['medium'] & rating['medium'] & discount['medium'], recommendation['medium'])

# 5. =======================
# Build system
# ==========================
recommendation_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6])
recommendation_sim = ctrl.ControlSystemSimulation(recommendation_ctrl)

# 6. =======================
# Uji pada beberapa produk
# ==========================
for i in range(5):
    item = data.iloc[i]
    recommendation_sim.input['price'] = item['price']
    recommendation_sim.input['rating'] = item['rating']
    recommendation_sim.input['popularity'] = item['popularity']
    recommendation_sim.input['discount'] = item['discount']

    recommendation_sim.compute()
    score = recommendation_sim.output['recommendation']

    print(f"{i+1}. {item['product_name']}")
    print(f"   Price: {item['price']}, Rating: {item['rating']}, Popularity: {item['popularity']}, Discount: {item['discount']}%")
    print(f"   => Fuzzy Recommendation Score: {score:.2f}")
    print("   => Category:", 
          "Highly Recommended" if score > 7 else 
          "Moderate" if score > 4 else "Not Recommended")
    print("-" * 60)


1. Running Shoes 170
   Price: 171958, Rating: 4.4, Popularity: 3822, Discount: 20%
   => Fuzzy Recommendation Score: 8.51
   => Category: Highly Recommended
------------------------------------------------------------
2. Action Figure 867
   Price: 1186074, Rating: 4.6, Popularity: 3221, Discount: 39%
   => Fuzzy Recommendation Score: 8.59
   => Category: Highly Recommended
------------------------------------------------------------
3. Smartphone 733
   Price: 837201, Rating: 4.2, Popularity: 2483, Discount: 20%
   => Fuzzy Recommendation Score: 8.42
   => Category: Highly Recommended
------------------------------------------------------------
4. Cycling Helmet 184
   Price: 1039436, Rating: 3.3, Popularity: 524, Discount: 41%
   => Fuzzy Recommendation Score: 5.09
   => Category: Moderate
------------------------------------------------------------
5. Air Purifier 657
   Price: 134654, Rating: 3.9, Popularity: 1949, Discount: 2%
   => Fuzzy Recommendation Score: 8.25
   => Category